# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a reproducible walkthrough for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library and Croissant’s schema-driven data interface.

### Dataset Source
The dataset is defined using a [Croissant schema](https://mlcommons.org/croissant/) at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We'll use `mlcroissant` to load both the schema metadata and the records.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_json = dataset.metadata.to_json()

print(f"Dataset Name: {getattr(dataset.metadata, 'name', '')}\n")
print(f"Description: {getattr(dataset.metadata, 'description', '')}\n")
if hasattr(dataset.metadata, 'dataCollection'):
    print(f"Data Collection: {dataset.metadata.dataCollection}\n")
if hasattr(dataset.metadata, 'dataLimitations'):
    print(f"Limitations: {dataset.metadata.dataLimitations}\n")
if hasattr(dataset.metadata, 'keywords'):
    print(f"Keywords: {dataset.metadata.keywords}\n")

## 2. Data Overview

Explore the record sets, their `@id`s and the fields/columns available for exploration.

In [ ]:
# List all record sets and their fields by @id

record_sets = dataset.record_sets
if len(record_sets) == 0:
    print("No record sets found in metadata. Trying to enumerate inferred/linked record sets...")
else:
    print(f"Number of record sets: {len(record_sets)}\n")

for record_set in record_sets:
    print(f"Record Set: {getattr(record_set, '@id', '[no @id]')}")
    # List available fields/columns in this record set
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {getattr(field, '@id', '')} (name: {getattr(field, 'name', '')}, type: {getattr(field, 'data_type', '')})")
    if hasattr(record_set, 'columns') and record_set.columns:
        print("  Columns:")
        for column in record_set.columns:
            print(f"    - {getattr(column, '@id', '')} (name: {getattr(column, 'name', '')}, type: {getattr(column, 'data_type', '')})")

if len(record_sets) == 0:
    # As a fallback, try dataset.record_set_ids
    record_set_ids = getattr(metadata_json, 'recordSet', []) if isinstance(metadata_json, dict) else []
    if record_set_ids:
        print("Record sets declared by @id in metadata:")
        print(record_set_ids)
    else:
        print("No record sets discovered. Please check the dataset schema.")

## 3. Data Extraction

Load data from record sets using the correct `@id` values, and convert them to Pandas DataFrames for further work. Use the output from the previous cell to pick available `@id`s.

**Note:** In FAIR² datasets, record sets often map to tables or CSVs used as primary data. All references are made using `@id`.

In [ ]:
# Gather all available record set @id values
record_set_ids = [getattr(rs, '@id', None) for rs in dataset.record_sets if getattr(rs, '@id', None) is not None]
dataframes = {}
for rs_id in record_set_ids:
    try:
        print(f"Loading records for record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Loaded {len(df)} rows. Columns:")
            print(f"    {df.columns.tolist()}")
        else:
            print(f"  No records found for {rs_id}.")
    except Exception as e:
        print(f"  Could not load records for {rs_id}: {e}")

# Show sample from the first loaded record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nSample records from record set: {first_rs_id}")
    display(dataframes[first_rs_id].head())
else:
    print("No tabular data loaded from record sets.")

## 4. Exploratory Data Analysis (EDA)

We'll process columns by `@id` (as available), filter numeric columns, remove outliers, normalize, and, if possible, group by a categorical field.

All data operations should use valid `@id`s from earlier steps.

In [ ]:
# Select a record set for EDA
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id].copy()
    print(f"Columns in {record_set_id}:")
    print(df.columns.tolist())
    # Try to find a numeric field @id for demonstration
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        print("No numeric field found for EDA. Please check field types.")
    else:
        print(f"Using numeric field '@id': {numeric_field}\n")
        # Remove rows with missing data in the field
        filtered_df = df[df[numeric_field].notnull()].copy()
        # Remove outliers beyond 3 std
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df = filtered_df[(filtered_df[numeric_field] >= mean - 3*std) & (filtered_df[numeric_field] <= mean + 3*std)]
        print(f"Filtered records (no missing, no strong outliers) for {numeric_field}:")
        print(filtered_df[[numeric_field]].head())
        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - mean) / std
        print(f"Normalized '{numeric_field}' preview:")
        print(filtered_df[[numeric_field, normalized_col]].head())
        # Find a group field @id (choose first suitable object/categorical)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object and df[col].nunique() < 30:
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by '{group_field}' (@id)")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
else:
    print("No loaded DataFrame to analyze.")

## 5. Visualization

Create visualizations of distribution and grouping relationships using fields referenced by `@id`. You may need to adjust field names if your specific dataset columns use different `@id` keys.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualizations using numeric_field, group_field if available
if 'filtered_df' in locals() and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()
    
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, preview, analyze, and visualize data from a FAIR² Croissant-based dataset using record sets, fields, and columns referenced by their `@id`. You can extend this workflow to further analyze adoption predictors, explore field-level statistics, and generate publication-ready figures. All entities were referenced by their Croissant `@id` for reproducibility.

For more information on the data, see the dataset documentation or the Croissant schema at the source URL.